## Лабораторная работа №1. Предобработка табличных данных

### (1) IEEE-CIS Fraud Detection — задача бинарной классификации мошеннических онлайн-транзакций.

Во время выполнения лабораторной работы были использованы следующие сторонние источники:
1. Информация о файловой системе - https://yandex.cloud/ru/docs/datasphere/operations/data/dataset?ysclid=mmq4tdjpmi88359202
2. Информация о датасете - https://www.kaggle.com/c/ieee-fraud-detection/discussion/101203

### 1. Импорт библиотек

Нам понадобятся библиотеки для работы с файловой системой ВМ, загрузки, просмотра и обработки датасетов, а также обучения модели классификации.

In [1]:
# работа с файловой системой
import os

# чистка traceback
import warnings

# работа с датасетами как с массивами и таблицами
import numpy as np
import pandas as pd

# предварительная обработка данных в датасетах
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from feature_engine.encoding import CountFrequencyEncoder

# базовая модель машинного обучения (дерево решений) для классификации
from sklearn.tree import DecisionTreeClassifier

# оценка точности предсказаний модели
from sklearn.metrics import accuracy_score

### 2. Загрузка набора данных ieee-fraud-detection

Узнаем, как называются файлы в датасете, найдем нужный нам файл с train data и посмотрим, как он выглядит в общих чертах.

In [2]:
# путь к датасету согласно документации Yandex Cloud
dataset_path = "/home/jupyter/datasets/ieee-fraud-detection"

# список всех файлов в датасете
print("Файлы в датасете:")
print(os.listdir(dataset_path))

# путь к train_public
file_path = os.path.join(dataset_path, "train_public.csv")
# загрузка csv-файла
df = pd.read_csv(file_path)

# все колонки
pd.set_option('display.max_columns', None)
# первые 10 строк
print("Набор тренировочных данных для предобработки и обучения:")
df.head()

Файлы в датасете:
['train_public.csv', 'test_public.csv', 'sample_submission.csv']
Набор тренировочных данных для предобработки и обучения:
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5   card6  addr1  addr2  dist1  dist2  \
0    NaN  150.0    discover  142.0  credit  315.0   87.0   19.0    NaN   
1  404.0  150.0  mastercard  102.0  credit  325.0   87.0    NaN    NaN   
2  490.0  150.0        visa  166.0   debit  330.0   87.0  287.0    NaN   
3  567.0  150.0  mastercard  117.0   debit  476.0   87.0    NaN    NaN   
4  514.0  150.0  mastercard

### 3. Предварительный анализ данных

Уже видно, что колонок много, а некоторые из них очень разрежены. Прежде чем писать трансформеры и строить пайплайн, посмотрим на типы признаков и их примерное содержимое, чтобы логически разбить признаки на подгруппы. Каждая подгруппа будет по-разному обрабатываться в пайплайне.

#### Общая структура данных и типы признаков

In [3]:
print("Shape:", df.shape)

print("\nТипы признаков:")
print(df.dtypes.value_counts())

Shape: (472432, 434)

Типы признаков:
float64    399
object      31
int64        4
Name: count, dtype: int64


#### Логическое разбиение признаков на подгруппы

1. Отдельно выносим целевую переменную isFraud. Тип int64, поэтому ей предобработка не нужна, она уже готова для обучения модели.
2. Временной признак time_features. Тип int64.
3. Числовые признаки numeric_features. Тип float64 и int64. В целом их можно разбить на две подгруппы:
- info_numeric_features - информационные числовые признаки, которые могут содержать выбросы, мешающие обучению (сумма транзакции и расстояния).
- monitor_numeric_features - мониторинговые числовые признаки, где либо семантически нет выбросов (различные номера и адреса), либо выбросы могут послужить сигналом мошенничества (C и D признаки).
4. Категориальные признаки categorical_features. Тип object.
5. Разреженные признаки sparse_features. Различные типы. Таким признакам лучше выделить свой imputer. В целом их можно разбить на три большие подгруппы:
- sparse_id_numeric_features - числовые id-признаки.
- sparse_id_categorical_features - категориальные id-признаки.
- sparse_V_features - числовые V-признаки.

In [4]:
# целевая переменная - является ли транзакция мошеннической
target = ["isFraud"]

# временной признак - Transaction DateTime
time_features = ["TransactionDT"]

# информативные числовые признаки
info_numeric_features = (
["TransactionAmt"]
+ ["dist1", "dist2"]
)
# мониторинговые числовые признаки
monitor_numeric_features = (
["TransactionID"]
+ ["card1","card2","card3","card5"]
+ ["addr1","addr2"]
+ [f"C{i}" for i in range(1,15)]
+ [f"D{i}" for i in range(1,16)]
)
# все числовые признаки
numeric_features = (
info_numeric_features +
monitor_numeric_features
)

# категориальные признаки
categorical_features = (
["ProductCD"]
+ ["card4","card6"]
+ ["P_emaildomain","R_emaildomain"]
+ ["DeviceType","DeviceInfo"]
+ [f"M{i}" for i in range(1,10)]
)

# все id_признаки
id_features = [c for c in df.columns if c.startswith("id_")]
# числовые id_признаки
sparse_id_numeric_features = df[id_features].select_dtypes(include=["int64","float64"]).columns.tolist()
# категориальные id_признаки
sparse_id_categorical_features = df[id_features].select_dtypes(include=["object"]).columns.tolist()
# V-признаки
sparse_V_features = [f"V{i}" for i in range(1,340)]
# все разреженные признаки
sparse_features = (
sparse_id_numeric_features +
sparse_id_categorical_features +
sparse_V_features
)

# все признаки
all_features = (
numeric_features +
categorical_features +
sparse_features +
time_features +
["isFraud"]
)

print("Все ли признаки были определены в подгруппу и нет ли пересечений? Ответ:", len(all_features) == len(df.columns))
print("\nТип целевой переменной:", df['isFraud'].dtype)
print("\nКоличество признаков во всех подгруппах (без учета таргета):")
print("Числовые:",len(numeric_features))
print("Категориальные:",len(categorical_features))
print("Временные:",len(time_features))
print("Разреженные (id_numeric):",len(sparse_id_numeric_features))
print("Разреженные (id_categorical):",len(sparse_id_categorical_features))
print("Разреженные (V):",len(sparse_V_features))

Все ли признаки были определены в подгруппу и нет ли пересечений? Ответ: True

Тип целевой переменной: int64

Количество признаков во всех подгруппах (без учета таргета):
Числовые: 39
Категориальные: 16
Временные: 1
Разреженные (id_numeric): 23
Разреженные (id_categorical): 15
Разреженные (V): 339


#### Анализ пропусков и аномалий

Узнаем топ признаков по доле пропусков в каждой подгруппе, выявим количество потенциальных аномалий в числовых признаках и посчитаем уникальные значения в категориальных признаках.

In [5]:
def analyze_missing_values(df, feature_groups):
    print("\n===== АНАЛИЗ ПРОПУСКОВ =====")
    for name, features in feature_groups.items():
        missing_ratio = df[features].isna().mean().sort_values(ascending=False)
        print(f"\n{name}")
        print(f"Количество признаков: {len(features)}")
        print("Топ признаков по доле пропусков:")
        print(missing_ratio.head(10))

def detect_numeric_outliers(df, numeric_features):
    print("\n===== АНОМАЛИИ В ЧИСЛОВЫХ ПРИЗНАКАХ =====")
    results = []
    for col in numeric_features:
        series = df[col].dropna()
        if len(series) == 0:
            continue
        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outliers = ((series < lower) | (series > upper)).sum()
        results.append({
            "feature": col,
            "outliers": outliers,
            "outlier_ratio": outliers / len(series)
        })
    results_df = pd.DataFrame(results).sort_values(
        "outlier_ratio",
        ascending=False
    )
    print(results_df.head(15))

def analyze_categorical_features(df, categorical_features):
    print("\n===== УНИКАЛЬНЫЕ ЗНАЧЕНИЯ КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ =====")
    for col in categorical_features:
        print(f"\n{col}")
        value_counts = df[col].value_counts(dropna=False)
        print("Количество уникальных значений:", df[col].nunique())

feature_groups = {
    "numeric": numeric_features,
    "categorical": categorical_features,
    "sparse": sparse_features,
    "time": time_features
}

analyze_missing_values(df, feature_groups)
detect_numeric_outliers(df, numeric_features)
analyze_categorical_features(df, categorical_features)


===== АНАЛИЗ ПРОПУСКОВ =====

numeric
Количество признаков: 39
Топ признаков по доле пропусков:
D7       0.936617
dist2    0.932054
D13      0.896339
D14      0.894512
D12      0.888443
D6       0.876209
D8       0.869590
D9       0.869590
dist1    0.607827
D5       0.537315
dtype: float64

categorical
Количество признаков: 16
Топ признаков по доле пропусков:
DeviceInfo       0.789311
R_emaildomain    0.759500
DeviceType       0.750995
M7               0.634970
M8               0.634957
M9               0.634957
M5               0.600116
M1               0.504401
M2               0.504401
M3               0.504401
dtype: float64

sparse
Количество признаков: 377
Топ признаков по доле пропусков:
id_24    0.991715
id_25    0.991012
id_07    0.990981
id_08    0.990981
id_21    0.990976
id_26    0.990968
id_27    0.990955
id_23    0.990955
id_22    0.990955
id_18    0.920789
dtype: float64

time
Количество признаков: 1
Топ признаков по доле пропусков:
TransactionDT    0.0
dtype: float64



Из анализа видно, что:
1. D-признаки, dist-ы, DeviceInfo, R_emaildomain и id-признаки являются очень разреженными.
2. В С и D-признаках зафиксировано много аномальных значений.
3. DeviceInfo и emaildomain-ы имеют слишком много уникальных значений и к ним точно не применим OneHot-Encoding.

### 4. Трансформеры

Подготовим трансформеры для каждой выделенной подгруппы по следующей схеме:
1. Преобразование типа данных в числовой или нормализация значений.
2. Обработка пропусков.
3. Обработка аномалий и выбросов.
4. Генерация новых информативных признаков, если это обосновано.

Трансформеры для числовых признаков.

In [6]:
# нормализация значений
# похоже, что мин-макс масштабирование нерелевантно - нет естественно ограниченных значений, поэтому берем стандартизацию
numeric_scaler = StandardScaler()

# обработка пропущенных значений
# для устойчивости к выбросам вставим в пропуски медиану
numeric_imputer = SimpleImputer(strategy='median')

# обработка аномалий
# по умолчанию воспользуемся методом межквартильного расстояния
class OutlierClipper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.q1 = np.nanpercentile(X, 25, axis=0)
        self.q3 = np.nanpercentile(X, 75, axis=0)
        self.iqr = self.q3 - self.q1
        return self
    def transform(self, X):
        X = X.copy()
        for i in range(X.shape[1]):
            if self.iqr[i] == 0:
                continue
            lower = self.q1[i] - 1.5 * self.iqr[i]
            upper = self.q3[i] + 1.5 * self.iqr[i]
            X[:, i] = np.clip(X[:, i], lower, upper)
        return X
numeric_outlier = OutlierClipper()

# генерация новых информативных признаков не является необходимой

Трансформеры для категориальных признаков.

In [7]:
# преобразование строк в числа
# похоже, что ординальных категорий нет, а OneHot может сильно увеличить размерность, поэтому реализуем FrequencyEncoder
categorical_encoder = CountFrequencyEncoder(encoding_method='frequency')

# обработка пропущенных значений
# вставим в пропуски моду, то есть самое частое значение
categorical_imputer = SimpleImputer(strategy="most_frequent")

# постобработка пропущенных значений
# вставим вместо невстреченных в train_public категорий частоту 0
categorical_post_imputer = SimpleImputer(strategy="constant", fill_value=0)

# обработка аномалий
# вообще говоря, FrequencyEncoder и так снижает влияние редких категорий

# генерация новых информативных признаков не является необходимой

Трансформеры для временных признаков.

In [8]:
# циклическое преобразование
# мошеннические транзакции скорее всего зависят от времени суток и дня недели, поэтому выделим именно их
class CyclicalTimeEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = np.array(X)  # ожидаем числовой timestamp в секундах
        hour = (X[:,0] // 3600) % 24
        day = X[:,0] // 86400
        weekday = day % 7
        hour_sin = np.sin(2 * np.pi * hour / 24)
        hour_cos = np.cos(2 * np.pi * hour / 24)
        weekday_sin = np.sin(2 * np.pi * weekday / 7)
        weekday_cos = np.cos(2 * np.pi * weekday / 7)
        return np.column_stack([hour_sin, hour_cos, weekday_sin, weekday_cos])
timestamp_encoder = CyclicalTimeEncoder()

# обработка пропущенных значений
# для устойчивости к выбросам вставим в пропуски медиану
timestamp_imputer = SimpleImputer(strategy='median')

# обработка аномалий
# по умолчанию воспользуемся методом межквартильного расстояния
timestamp_outlier = OutlierClipper()

# генерация новых информативных признаков проводится в CyclicalTimeEncoder - час и день недели

Трансформеры для разреженных числовых признаков.

In [9]:
# нормализация значений
# похоже, что мин-макс масштабирование нерелевантно - нет естественно ограниченных значений, поэтому берем стандартизацию
sparse_numeric_scaler = StandardScaler()

# обработка пропущенных значений
# слишком много NaN, впишем редкую константу как индикатор пропущенного значения
sparse_numeric_imputer = SimpleImputer(strategy="constant", fill_value=-999)

# обработка аномалий
# скорее всего разрушит структуру и так очень разреженных признаков

# генерация новых информативных признаков не является необходимой

Трансформеры для разреженных категориальных признаков.

In [10]:
# преобразование строк в числа
# похоже, что ординальных категорий нет, а OneHot может сильно увеличить размерность, поэтому реализуем FrequencyEncoder
sparse_categorical_encoder = CountFrequencyEncoder(encoding_method='frequency')

# обработка пропущенных значений
# слишком много NaN, впишем константу как индикатор пропущенного значения
sparse_categorical_imputer = SimpleImputer(strategy="constant", fill_value="missing")

# постобработка пропущенных значений
# вставим вместо невстреченных в train_public категорий частоту 0
sparse_categorical_post_imputer = SimpleImputer(strategy="constant", fill_value=0)

# обработка аномалий
# скорее всего разрушит структуру и так очень разреженных признаков

# генерация новых информативных признаков не является необходимой

### 5. Пайплайны обработки

Соберем написанные трансформеры в пайплайны для каждой подгруппы признаков.

Пайплайн обработки числовых признаков.

In [11]:
# выкидываем мешающие выбросы outlier-ом
info_numeric_pipeline = Pipeline([
    ("imputer", numeric_imputer),
    ("outliers", numeric_outlier),
    ("scaler", numeric_scaler)
])

# не выкидываем выбросы из номеров, адресов и сигнальных C и D-признаков
monitor_numeric_pipeline = Pipeline([
    ("imputer", numeric_imputer),
    ("scaler", numeric_scaler)
])

Пайплайн обработки категориальных признаков.

In [12]:
categorical_pipeline = Pipeline([
    ("imputer", categorical_imputer),
    ("encoder", categorical_encoder),
    ("post_imputer", categorical_post_imputer)
])

Пайплайн обработки временных признаков.

In [13]:
time_pipeline = Pipeline([
    ("imputer", timestamp_imputer),
    ("outliers", timestamp_outlier),
    ("cyclical", timestamp_encoder)
])

Пайплайн обработки разреженных числовых признаков.

In [14]:
sparse_numeric_pipeline = Pipeline([
    ("imputer", sparse_numeric_imputer),
    ("scaler", sparse_numeric_scaler)
])

Пайплайн обработки разреженных категориальных признаков.

In [15]:
sparse_categorical_pipeline = Pipeline([
    ("imputer", sparse_categorical_imputer),
    ("encoder", sparse_categorical_encoder),
    ("post_imputer", sparse_categorical_post_imputer)
])

### 6. Общий пайплайн

In [16]:
preprocessor = ColumnTransformer([
    ("info_numeric", info_numeric_pipeline, info_numeric_features),
    ("monitor_numeric", monitor_numeric_pipeline, monitor_numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
    ("time", time_pipeline, time_features),
    ("sparse_numeric", sparse_numeric_pipeline,
        sparse_id_numeric_features + sparse_V_features),
    ("sparse_categorical", sparse_categorical_pipeline, sparse_id_categorical_features)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor)
])

### 7. Применение к train_public

Обработка train_public созданным пайплайном и результат.

In [17]:
# чистка traceback, возникающего из-за вызова CountFrequencyEncoder из feature_engine.encoding
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="sklearn.pipeline",
    message="This Pipeline instance is not fitted yet"
)

# разделение train_public на признаки и целевую переменную isFraud
X = df.drop(columns=["isFraud"])
y = df["isFraud"]

# предобработка данных построенным пайплайном
X_processed = pipeline.fit_transform(X)

print("Shape после обработки:", X_processed.shape)
print("Результат обработки train_public:")
print(X_processed[:5])

Shape после обработки: (472432, 436)
Результат обработки train_public:
[[-0.40698565 -0.1412148  -0.09119928 ...  0.75066041  0.75066041
   0.75066041]
 [-0.94372254 -0.18262432 -0.09119928 ...  0.75066041  0.75066041
   0.75066041]
 [-0.53607427  0.96856038 -0.09119928 ...  0.75066041  0.75066041
   0.75066041]
 [-0.65836875 -0.18262432 -0.09119928 ...  0.75066041  0.75066041
   0.75066041]
 [-0.65836875 -0.18262432 -0.09119928 ...  0.23643403  0.19489154
   0.12778982]]


Более наглядное представление результата - таблица с именованными колонками и статистика.

In [18]:
processed_columns = []
processed_columns += info_numeric_features
processed_columns += monitor_numeric_features
processed_columns += categorical_features
processed_columns += ["hour_sin", "hour_cos", "weekday_sin", "weekday_cos"]
processed_columns += sparse_id_numeric_features + sparse_V_features
processed_columns += sparse_id_categorical_features

print("Таблица обработанных данных:")
X_processed_df = pd.DataFrame(X_processed, columns=processed_columns)
X_processed_df.head()

Таблица обработанных данных:


,TransactionAmt,dist1,dist2,TransactionID,card1,card2,card3,card5,addr1,addr2,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,ProductCD,card4,card6,P_emaildomain,R_emaildomain,DeviceType,DeviceInfo,M1,M2,M3,M4,M5,M6,M7,M8,M9,hour_sin,hour_cos,weekday_sin,weekday_cos,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_13,id_14,id_17,id_18,id_19,id_20,id_21,id_22,id_24,id_25,id_26,id_32,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30,V31,V32,V33,V34,V35,V36,V37,V38,V39,V40,V41,V42,V43,V44,V45,V46,V47,V48,V49,V50,V51,V52,V53,V54,V55,V56,V57,V58,V59,V60,V61,V62,V63,V64,V65,V66,V67,V68,V69,V70,V71,V72,V73,V74,V75,V76,V77,V78,V79,V80,V81,V82,V83,V84,V85,V86,V87,V88,V89,V90,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100,V101,V102,V103,V104,V105,V106,V107,V108,V109,V110,V111,V112,V113,V114,V115,V116,V117,V118,V119,V120,V121,V122,V123,V124,V125,V126,V127,V128,V129,V130,V131,V132,V133,V134,V135,V136,V137,V138,V139,V140,V141,V142,V143,V144,V145,V146,V147,V148,V149,V150,V151,V152,V153,V154,V155,V156,V157,V158,V159,V160,V161,V162,V163,V164,V165,V166,V167,V168,V169,V170,V171,V172,V173,V174,V175,V176,V177,V178,V179,V180,V181,V182,V183,V184,V185,V186,V187,V188,V189,V190,V191,V192,V193,V194,V195,V196,V197,V198,V199,V200,V201,V202,V203,V204,V205,V206,V207,V208,V209,V210,V211,V212,V213,V214,V215,V216,V217,V218,V219,V220,V221,V222,V223,V224,V225,V226,V227,V228,V229,V230,V231,V232,V233,V234,V235,V236,V237,V238,V239,V240,V241,V242,V243,V244,V245,V246,V247,V248,V249,V250,V251,V252,V253,V254,V255,V256,V257,V258,V259,V260,V261,V262,V263,V264,V265,V266,V267,V268,V269,V270,V271,V272,V273,V274,V275,V276,V277,V278,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V305,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339,id_12,id_15,id_16,id_23,id_27,id_28,id_29,id_30,id_31,id_33,id_34,id_35,id_36,id_37,id_38
0,-0.406986,-0.141215,-0.091199,-1.732047,0.828333,-0.011687,-0.287298,-1.416093,0.245538,0.074463,-0.097229,-0.091947,-0.035851,-0.062322,-0.215513,-0.111971,-0.05055,-0.057260,-0.204738,-0.058258,-0.087797,-0.051269,-0.235495,-0.144633,-0.499408,-0.278679,-0.130561,-0.486952,-0.241065,-0.149106,-0.101184,-0.152294,0.115125,-0.539396,-0.535145,-0.13431,-0.080075,-0.121159,-0.756730,0.732383,0.011418,0.257637,0.542465,0.859599,0.902435,0.873305,0.999956,0.948924,0.895526,0.103249,0.820958,0.321773,0.950626,0.865811,0.943429,0.0,1.0,0.781831,0.62349,-0.584948,-0.401617,-0.359578,-0.359578,-0.565538,-0.565445,-0.095395,-0.095365,-0.387256,-0.387255,-0.576333,-0.535089,-0.405847,-0.572466,-0.293301,-0.568546,-0.568148,-0.094418,-0.095534,-0.0914,-0.094965,-0.095431,-0.403362,1.042195,1.042115,1.042052,1.042512,1.042451,1.042113,1.042061,1.042147,1.042121,1.041270,1.041241,0.405810,0.405697,0.404490,0.404128,0.404124,0.404101,0.404098,0.405018,0.404926,0.404114,0.404107,0.404388,0.404322,0.404560,0.404530,0.404487,0.404487,0.403376,0.403324,0.404080,0.404075,0.404108,0.404089,-1.542440,-1.542440,-1.542440,-1.542439,-1.542440,-1.542440,-1.542441,-1.542441,-1.542441,-1.542440,-1.542439,-1.542441,-1.542441,-1.542440,-1.54244,-1.542441,-1.542441,-1.542441,0.407405,0.407284,0.405935,0.405802,0.405747,0.405740,0.405739,0.405719,0.406620,0.406514,0.405750,0.405722,0.406129,0.406191,0.406142,0.406126,0.405013,0.404963,0.405719,0.405710,0.405719,0.405689,0.445267,0.445151,0.443748,0.443615,0.443611,0.443587,0.443568,0.441710,0.441612,0.443607,0.443580,0.443809,0.443718,0.443989,0.443985,0.442909,0.442860,0.443568,0.443561,0.443609,-0.00096,-0.004458,-0.020603,0.01018,-0.040610,-0.002674,0.007324,0.058833,0.001861,0.010276,0.005456,0.008818,0.014344,0.014003,0.01336,0.013812,0.014131,0.013953,0.014082,0.0137,0.012263,0.013306,0.014287,0.014229,0.0142

In [19]:
print("Статистические данные:")
X_processed_df.describe()

Статистические данные:


,TransactionAmt,dist1,dist2,TransactionID,card1,card2,card3,card5,addr1,addr2,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,ProductCD,card4,card6,P_emaildomain,R_emaildomain,DeviceType,DeviceInfo,M1,M2,M3,M4,M5,M6,M7,M8,M9,hour_sin,hour_cos,weekday_sin,weekday_cos,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_13,id_14,id_17,id_18,id_19,id_20,id_21,id_22,id_24,id_25,id_26,id_32,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30,V31,V32,V33,V34,V35,V36,V37,V38,V39,V40,V41,V42,V43,V44,V45,V46,V47,V48,V49,V50,V51,V52,V53,V54,V55,V56,V57,V58,V59,V60,V61,V62,V63,V64,V65,V66,V67,V68,V69,V70,V71,V72,V73,V74,V75,V76,V77,V78,V79,V80,V81,V82,V83,V84,V85,V86,V87,V88,V89,V90,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100,V101,V102,V103,V104,V105,V106,V107,V108,V109,V110,V111,V112,V113,V114,V115,V116,V117,V118,V119,V120,V121,V122,V123,V124,V125,V126,V127,V128,V129,V130,V131,V132,V133,V134,V135,V136,V137,V138,V139,V140,V141,V142,V143,V144,V145,V146,V147,V148,V149,V150,V151,V152,V153,V154,V155,V156,V157,V158,V159,V160,V161,V162,V163,V164,V165,V166,V167,V168,V169,V170,V171,V172,V173,V174,V175,V176,V177,V178,V179,V180,V181,V182,V183,V184,V185,V186,V187,V188,V189,V190,V191,V192,V193,V194,V195,V196,V197,V198,V199,V200,V201,V202,V203,V204,V205,V206,V207,V208,V209,V210,V211,V212,V213,V214,V215,V216,V217,V218,V219,V220,V221,V222,V223,V224,V225,V226,V227,V228,V229,V230,V231,V232,V233,V234,V235,V236,V237,V238,V239,V240,V241,V242,V243,V244,V245,V246,V247,V248,V249,V250,V251,V252,V253,V254,V255,V256,V257,V258,V259,V260,V261,V262,V263,V264,V265,V266,V267,V268,V269,V270,V271,V272,V273,V274,V275,V276,V277,V278,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V305,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339,id_12,id_15,id_16,id_23,id_27,id_28,id_29,id_30,id_31,id_33,id_34,id_35,id_36,id_37,id_38
count,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e+05,4.724320e

### 8. Обучение модели классификации

В качестве базовой модели машинного обучения для задачи классификации возьмем дерево решений, потому что признаков много, а дерево решений автоматически выбирает наиболее информативные признаки и игнорирует несущественные.

In [20]:
# параметры модели
model = DecisionTreeClassifier(
    max_depth=12,           # глубина дерева
    min_samples_split=10,   # минимальное число объектов для разбиения
    min_samples_leaf=5,     # минимальное число объектов в листе
    max_features='sqrt',    # случайный набор признаков для разбиения
    class_weight='balanced' # учитываем дисбаланс классов
)

# обучение модели на обработанных данных из train_public
model.fit(X_processed_df, y)

,criterion,'gini'
,splitter,'best'
,max_depth,12
,min_samples_split,10
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,'balanced'


### 9. Получение предсказаний от обученной модели

Загрузка test data, их предобработка и результат.

In [21]:
# чистка warnings, возникающих из-за вызова CountFrequencyEncoder из feature_engine.encoding
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="feature_engine.encoding",
    message="During the encoding, NaN values were introduced"
)
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="sklearn.pipeline",
    message="This Pipeline instance is not fitted yet"
)

# путь к test_public
test_file_path = os.path.join(dataset_path, "test_public.csv")
# загрузка csv-файла
test_df = pd.read_csv(test_file_path)

# предобработка данных из test_public
X_test_processed = pipeline.transform(test_df)
X_test_df = pd.DataFrame(X_test_processed, columns=processed_columns)
print("Результат обработки test_public:")
X_test_df.head()

Результат обработки test_public:


,TransactionAmt,dist1,dist2,TransactionID,card1,card2,card3,card5,addr1,addr2,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,ProductCD,card4,card6,P_emaildomain,R_emaildomain,DeviceType,DeviceInfo,M1,M2,M3,M4,M5,M6,M7,M8,M9,hour_sin,hour_cos,weekday_sin,weekday_cos,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_13,id_14,id_17,id_18,id_19,id_20,id_21,id_22,id_24,id_25,id_26,id_32,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30,V31,V32,V33,V34,V35,V36,V37,V38,V39,V40,V41,V42,V43,V44,V45,V46,V47,V48,V49,V50,V51,V52,V53,V54,V55,V56,V57,V58,V59,V60,V61,V62,V63,V64,V65,V66,V67,V68,V69,V70,V71,V72,V73,V74,V75,V76,V77,V78,V79,V80,V81,V82,V83,V84,V85,V86,V87,V88,V89,V90,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100,V101,V102,V103,V104,V105,V106,V107,V108,V109,V110,V111,V112,V113,V114,V115,V116,V117,V118,V119,V120,V121,V122,V123,V124,V125,V126,V127,V128,V129,V130,V131,V132,V133,V134,V135,V136,V137,V138,V139,V140,V141,V142,V143,V144,V145,V146,V147,V148,V149,V150,V151,V152,V153,V154,V155,V156,V157,V158,V159,V160,V161,V162,V163,V164,V165,V166,V167,V168,V169,V170,V171,V172,V173,V174,V175,V176,V177,V178,V179,V180,V181,V182,V183,V184,V185,V186,V187,V188,V189,V190,V191,V192,V193,V194,V195,V196,V197,V198,V199,V200,V201,V202,V203,V204,V205,V206,V207,V208,V209,V210,V211,V212,V213,V214,V215,V216,V217,V218,V219,V220,V221,V222,V223,V224,V225,V226,V227,V228,V229,V230,V231,V232,V233,V234,V235,V236,V237,V238,V239,V240,V241,V242,V243,V244,V245,V246,V247,V248,V249,V250,V251,V252,V253,V254,V255,V256,V257,V258,V259,V260,V261,V262,V263,V264,V265,V266,V267,V268,V269,V270,V271,V272,V273,V274,V275,V276,V277,V278,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V305,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339,id_12,id_15,id_16,id_23,id_27,id_28,id_29,id_30,id_31,id_33,id_34,id_35,id_36,id_37,id_38
0,-0.885823,-0.182624,-0.091199,1.732054,-0.115911,-1.656138,2.760854,-1.514142,0.078780,0.074463,-0.097229,-0.091947,-0.035851,-0.048885,-0.215513,-0.111971,-0.036053,-0.047834,-0.265525,-0.048858,-0.097515,-0.040942,-0.235495,-0.144633,-0.590846,-0.278679,-0.237014,-0.621622,-0.241065,-0.149106,-0.101184,-0.152294,0.115125,-0.616971,-0.370415,-0.13431,-0.080075,-0.121159,-0.756730,0.119403,0.654007,0.742268,0.542465,0.859599,0.902435,0.873305,0.999956,0.948924,0.895526,0.103249,0.820958,0.678227,0.950626,0.865811,0.943429,0.5,0.866025,0.781831,0.62349,1.720512,1.248874,-0.359578,-0.359578,1.766465,1.783465,-0.095395,-0.095365,-0.387256,-0.387255,1.735664,1.906457,-0.405847,1.816176,3.412433,1.862881,1.758750,-0.094418,-0.095534,-0.0914,-0.094965,-0.095431,-0.403362,-0.959513,-0.959513,-0.959513,-0.959513,-0.959513,-0.959513,-0.959513,-0.959513,-0.959513,-0.959513,-0.959513,0.402932,0.402819,0.404490,0.407007,0.407003,0.406981,0.406977,0.405018,0.404926,0.406994,0.406987,0.404388,0.404322,0.404560,0.404530,0.404487,0.404487,0.403376,0.403324,0.406960,0.406954,0.406987,0.406968,0.647182,0.647102,0.648079,0.647966,0.65014,0.650116,0.648325,0.650163,0.650136,0.648131,0.648052,0.648272,0.648236,0.647492,0.64746,0.650141,0.650132,0.650108,0.404536,0.404414,0.405935,0.405802,0.408618,0.408610,0.408610,0.408589,0.406620,0.406514,0.408621,0.408593,0.406129,0.406191,0.406142,0.406126,0.405013,0.404963,0.408590,0.408581,0.408590,0.408560,0.442569,0.442454,0.443748,0.443615,0.446309,0.446286,0.446266,0.444406,0.444308,0.446306,0.446279,0.443809,0.443718,0.443989,0.443985,0.442909,0.442860,0.446267,0.446259,0.446308,-0.00096,-0.071139,-0.020603,0.01018,-0.04061,-0.002674,0.007324,-0.009808,0.001861,0.010276,0.005456,0.008818,0.014344,0.014003,0.01336,0.013812,0.014131,0.013953,0.014082,0.0137,0.012263,0.013306,0.014287,0.014229,0.01427,0.014259,0.

Предсказания обученной модели.

In [22]:
test_pred = model.predict_proba(X_test_df)[:,1]

submission = pd.DataFrame({
    "TransactionID": test_df["TransactionID"],
    "isFraud": test_pred
})

print("Посмотрим на вычисленные вероятности:")
submission.head(10)

Посмотрим на вычисленные вероятности:


,TransactionID,isFraud
0,3459432,0.346217
1,3459433,0.065096
2,3459434,0.840726
3,3459435,0.831737
4,3459436,0.239196
5,3459437,0.165303
6,3459438,0.000000
7,3459439,0.210432
8,3459440,0.210432
9,3459441,0.144064


In [23]:
threshold = 0.5
test_pred_flag = (test_pred >= threshold).astype(float)

submission_hard = pd.DataFrame({
    "TransactionID": test_df["TransactionID"],
    "isFraud": test_pred_flag
})

print("Жесткая классификация как в sample_submission.csv:")
submission_hard.head(10)

Жесткая классификация как в sample_submission.csv:


,TransactionID,isFraud
0,3459432,0.0
1,3459433,0.0
2,3459434,1.0
3,3459435,1.0
4,3459436,0.0
5,3459437,0.0
6,3459438,0.0
7,3459439,0.0
8,3459440,0.0
9,3459441,0.0


### 10. Оценка предсказаний обученной модели

Проверим, насколько точно обученная модель разделила транзакции на мошеннические и не мошеннические. Для этого сравним ее предсказания с реальными значениями таргета из sample_submission.

In [24]:
# загрузка настоящих значений таргета
target_file_path = os.path.join(dataset_path, "sample_submission.csv")
target_df = pd.read_csv(target_file_path)

# выделяем колонку с таргетом
target_true = target_df["isFraud"]

# общий accuracy
acc_total = accuracy_score(target_true, test_pred_flag)
print("Общая точность предсказаний модели:", acc_total)

# accuracy для обычных транзакций (0)
mask_normal = target_true == 0
if mask_normal.sum() > 0:
    acc_normal = accuracy_score(target_true[mask_normal], test_pred_flag[mask_normal])
    print("Точность предсказаний для обычных транзакций:", acc_normal)
else:
    print("Обычных транзакций в таргете нет")

# accuracy для мошеннических транзакций (1)
mask_fraud = target_true == 1
if mask_fraud.sum() > 0:
    acc_fraud = accuracy_score(target_true[mask_fraud], test_pred_flag[mask_fraud])
    print("Точность предсказаний для мошеннических транзакций:", acc_fraud)
else:
    print("Мошеннических транзакций в таргете нет")

Общая точность предсказаний модели: 0.8298083110373556
Точность предсказаний для обычных транзакций: 0.8298083110373556
Мошеннических транзакций в таргете нет


Полученная точность (~0.83) приемлема для такой базовой модели, как дерево решений. Обученная модель отнесла некоторые транзакции к мошенническим, несмотря на то что таких там не было.

## (2) NYC — задача регрессии длительности поездок

Во время выполнения лабораторной работы были использованы сторонние источники:
1. Информация о датасете - https://www.kaggle.com/datasets/usmanshams/nyc-yellow-taxi-dataset-2024/data

### 1. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

### 2. Загрузка датасета

Загрузим датасет и рассмотрим его структуру

In [2]:
train_features = pd.read_csv("/home/jupyter/datasets/nyc-taxi-small-public/X_train_public.csv")
train_target = pd.read_csv("/home/jupyter/datasets/nyc-taxi-small-public/y_train_public.csv")
val_features = pd.read_csv("/home/jupyter/datasets/nyc-taxi-small-public/X_test_public.csv")

train_features['trip_duration']=train_target

print(f"Размер обучающих признаков: {train_features.shape}")
print(f"Размер целевой переменной: {train_target.shape}")
print(f"Размер тестовых признаков: {val_features.shape}")

Размер обучающих признаков: (9551977, 19)
Размер целевой переменной: (9551977, 1)
Размер тестовых признаков: (3513200, 18)


In [3]:
train_features.head(10)

,VendorID,tpep_pickup_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,trip_duration
0,2,2024-01-01 00:57:55,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.00,1188.0
1,1,2024-01-01 00:03:00,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.00,396.0
2,1,2024-01-01 00:17:06,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.00,1075.0
3,1,2024-01-01 00:36:38,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.00,498.0
4,1,2024-01-01 00:46:51,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.00,366.0
5,1,2024-01-01 00:54:08,1.0,4.70,1.0,N,148,141,1,29.6,3.5,0.5,6.90,0.0,1.0,41.50,2.5,0.00,1943.0
6,2,2024-01-01 00:49:44,2.0,10.82,1.0,N,138,181,1,45.7,6.0,0.5,10.00,0.0,1.0,64.95,0.0,1.75,1563.0
7,1,2024-01-01 00:30:40,0.0,3.00,1.0,N,246,231,2,25.4,3.5,0.5,0.00,0.0,1.0,30.40,2.5,0.00,1680.0
8,2,2024-01-01 00:26:01,1.0,5.44,1.0,N,161,261,2,31.0,1.0,0.5,0.00,0.0,1.0,36.00,2.5,0.00,1691.0
9,2,2024-01-01 00:28:08,1.0,0.04,1.0,N,113,113,2,3.0,1.0,0.5,0.00,0.0,1.0,8.00,2.5,0.00,68.0


### 3. Анализ датасета

In [4]:
print("\nПропущенные значения:\n", train_features.isnull().sum())


Пропущенные значения:
 VendorID                      0
tpep_pickup_datetime          0
passenger_count          751524
trip_distance                 0
RatecodeID               751524
store_and_fwd_flag       751524
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     751524
Airport_fee              751524
trip_duration                 0
dtype: int64


NaN значения появляются только в 5 колонках. Во всех них эти значения можно заменить на наиболее часто встречающееся

In [5]:
print("Количество нулей\n")
def check_for_zeros(train_features):
    for col in train_features.columns:
        zeros = train_features[train_features[col] == 0].shape[0]
        print(f"Нулей в {col}:{zeros}")
        
check_for_zeros(train_features)

Количество нулей

Нулей в VendorID:0
Нулей в tpep_pickup_datetime:0
Нулей в passenger_count:105702
Нулей в trip_distance:213308
Нулей в RatecodeID:0
Нулей в store_and_fwd_flag:0
Нулей в PULocationID:0
Нулей в DOLocationID:0
Нулей в payment_type:751524
Нулей в fare_amount:2779
Нулей в extra:4269510
Нулей в mta_tax:96119
Нулей в tip_amount:2511932
Нулей в tolls_amount:8870815
Нулей в improvement_surcharge:4871
Нулей в total_amount:1093
Нулей в congestion_surcharge:660545
Нулей в Airport_fee:8074983
Нулей в trip_duration:0


В колонках, наиболее подходящих для предсказания времения поездки, нулевые значения встречаются в количестве, которым можно пренебречь. 
Было принято решение откинуть строки, в которых значения определенно ошибочные. 

In [6]:
print("\nОписательная статистика:")
print(train_features[['trip_distance', 'fare_amount', 'trip_duration','tip_amount', 'tolls_amount']].describe())


Описательная статистика:
       trip_distance   fare_amount  trip_duration    tip_amount  tolls_amount
count   9.551977e+06  9.551977e+06   9.551977e+06  9.551977e+06  9.551977e+06
mean    4.043286e+00  1.832209e+01   9.678822e+02  3.271801e+00  5.263650e-01
std     2.655172e+02  1.834269e+01   2.060235e+03  3.927542e+00  2.124263e+00
min     0.000000e+00 -9.990000e+02   1.000000e+00 -3.000000e+02 -8.430000e+01
25%     1.000000e+00  8.600000e+00   4.430000e+02  0.000000e+00  0.000000e+00
50%     1.700000e+00  1.280000e+01   7.240000e+02  2.650000e+00  0.000000e+00
75%     3.190000e+00  2.050000e+01   1.161000e+03  4.120000e+00  0.000000e+00
max     3.127223e+05  9.792000e+03   5.673240e+05  9.999900e+02  1.630000e+02


Предполагаем, что поездка на такси не может быть на расстояние в 30000 миль,  стоимостью 10000 долларов  и длительностью 158 часов.

In [7]:
print(train_features.VendorID.value_counts(dropna=False), '\n')
print(train_features.payment_type.value_counts(dropna=False), '\n')
print(train_features.RatecodeID.value_counts(dropna=False), '\n')
print(train_features.store_and_fwd_flag.value_counts(dropna=False), '\n')

VendorID
2    7219502
1    2331909
6        566
Name: count, dtype: int64 

payment_type
1    7257818
2    1328187
0     751524
4     152543
3      61905
Name: count, dtype: int64 

RatecodeID
1.0     8292344
NaN      751524
2.0      307650
99.0      96538
5.0       59236
3.0       25380
4.0       19288
6.0          17
Name: count, dtype: int64 

store_and_fwd_flag
N      8764509
NaN     751524
Y        35944
Name: count, dtype: int64 



При анализе категориальных признаков наблюдается значительный дисбаланс классов: одно из значений встречается гораздо чаще остальных. В связи с этим при заполнении пропусков наиболее частотное значение используется в качестве значения по умолчанию.

In [8]:
print(train_features.extra.value_counts(dropna=False), '\n')
print(train_features.mta_tax.value_counts(dropna=False), '\n')
print(train_features.improvement_surcharge.value_counts(dropna=False), '\n')
print(train_features.congestion_surcharge.value_counts(dropna=False), '\n')
print(train_features.Airport_fee.value_counts(dropna=False), '\n')

extra
0.00    4269510
2.50    2168504
1.00    1707623
5.00     602248
3.50     466231
         ...   
2.45          1
5.75          1
0.13          1
0.10          1
1.05          1
Name: count, Length: 82, dtype: int64 

mta_tax
 0.50     9343418
-0.50      112395
 0.00       96119
 1.00          16
 4.00          14
 0.16           4
 3.00           4
-0.16           3
 1.60           1
 1.50           1
 35.84          1
 5.75           1
Name: count, dtype: int64 

improvement_surcharge
 1.0    9429623
-1.0     115872
 0.0       4871
 0.3       1608
-0.3          3
Name: count, dtype: int64 

congestion_surcharge
 2.50    8044785
 NaN      751524
 0.00     660545
-2.50      95111
 0.75          5
 1.00          5
-0.75          2
Name: count, dtype: int64 

Airport_fee
 0.00    8074983
 NaN      751524
 1.75     709201
-1.75      16269
Name: count, dtype: int64 



Поскольку пайплайны не предназначены для изменения количества строк в датасете, предварительная очистка данных выполняется до их применения. 
На этом этапе удаляются ошибочные записи, которые могут негативно повлиять на обучение модели и которые проще отбросить, чем корректировать.

In [9]:
train_features = train_features[(train_features.trip_distance >= 0)&(train_features.trip_distance.notnull())&(train_features.trip_distance <= 120)]
train_features = train_features[(train_features.fare_amount > 0)&(train_features.fare_amount <= 200)]
train_features = train_features[(train_features.passenger_count > 0)&(train_features.passenger_count<=6)]
train_features = train_features[(train_features.trip_duration>0)&(train_features.trip_duration<14400)]

In [10]:
train_target = train_features.trip_duration

In [11]:
train_features=train_features.drop(columns=['trip_duration'],axis=1)

### 4. Определение типов признаков

In [12]:
# Непрерывные числовые признаки (требуют масштабирования)
continuous_features = ['trip_distance', 'fare_amount', 'tip_amount', 
                       'tolls_amount', 'total_amount', 
                      'congestion_surcharge',]

# Дискретные числовые признаки (целые числа)
discrete_features = ['passenger_count'] 

# Фиксированные надбавки (ordinal encoding)
surcharge_features = ['extra', 'mta_tax', 'improvement_surcharge', 'Airport_fee', 'store_and_fwd_flag']

# Категориальные признаки с малым числом категорий
categorical_small = ['VendorID', 'RatecodeID', 'payment_type']

# Категориальные признаки с большим числом категорий (локации)
categorical_large = ['PULocationID', 'DOLocationID']

# Временные признаки
datetime_features = ['tpep_pickup_datetime']

### 5. Трансформеры

In [13]:
class DateTimeExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        X['tpep_pickup_datetime'] = pd.to_datetime(X['tpep_pickup_datetime'])
        
        # Основные временные признаки
        X['pickup_dayofweek'] = X['tpep_pickup_datetime'].dt.dayofweek
        X['pickup_hour'] = X['tpep_pickup_datetime'].dt.hour
        X['pickup_month'] = X['tpep_pickup_datetime'].dt.month
        X['pickup_day'] = X['tpep_pickup_datetime'].dt.day
        
        # Циклическое кодирование времени
        seconds_in_day = 24 * 60 * 60
        time_of_day = (X['tpep_pickup_datetime'].dt.hour * 3600 + 
                       X['tpep_pickup_datetime'].dt.minute * 60 + 
                       X['tpep_pickup_datetime'].dt.second) / seconds_in_day
        time_of_day_rad = 2 * np.pi * time_of_day
        X['time_sin'] = np.sin(time_of_day_rad)
        X['time_cos'] = np.cos(time_of_day_rad)
        
        # Удаляем исходный столбец
        return X.drop(columns=['tpep_pickup_datetime'])

Поскольку все записи в датасете относятся к первым трем месяцам 2024 года, признаки года и дня года (включая их циклическое кодирование) не несут полезной информации. Предполагается, что для предсказания загруженности дорог и длительности поездки наиболее значимыми будут признаки дня недели и циклически закодированного времени суток.

In [14]:
class LocationFrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.location_freq_ = None
        
    def fit(self, X, y=None):
        # Объединяются все локации из обоих столбцов
        all_locations = pd.concat([X['PULocationID'], X['DOLocationID']])
        # Считается частоту каждого location ID
        self.location_freq_ = all_locations.value_counts(normalize=True).to_dict()
        return self
    
    def transform(self, X):
        X = X.copy()
        # Создаем строковый идентификатор пары локаций
        X['location_pair'] = X['PULocationID'].astype(str) + '_' + X['DOLocationID'].astype(str)
        
        # Частотное кодирование для пары
        pair_freq = X['location_pair'].value_counts(normalize=True).to_dict()
        X['location_pair_freq'] = X['location_pair'].map(pair_freq)
        
        # Также добавляем частотное кодирование для отдельных локаций
        X['PU_freq'] = X['PULocationID'].map(self.location_freq_)
        X['DO_freq'] = X['DOLocationID'].map(self.location_freq_)
        
        # Удаляем исходные колонки и промежуточные
        return X.drop(columns=['PULocationID', 'DOLocationID', 'location_pair'])

One-hot кодирование для полей PULocationID и DOLocationID (по ~260 уникальных значений каждое) нецелесообразно из-за взрывного роста размерности. Вместо этого каждая комбинация посадки и высадки была преобразована в уникальную строку и закодирована частотой встречаемости.

### 6. Создание пайплайнов

In [24]:
#обработчик аномалий до основных трансформеров
continuous_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', RobustScaler())
])

# Дискретные признаки: SimpleImputer с константой 1
discrete_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=1)),  
])

surcharge_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        handle_unknown='use_encoded_value', 
        unknown_value=-1                      
    ))
])

# Малые категориальные: SimpleImputer + OneHotEncoder
categorical_small_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Временные признаки: кастомный DateTimeExtractor
datetime_pipeline = Pipeline([
    ('extractor', DateTimeExtractor()),
])

# Локации: кастомный LocationFrequencyEncoder
location_pipeline = Pipeline([
    ('encoder', LocationFrequencyEncoder())
   
])

Для непрерывных признаков использован RobustScaler - он менее чувствителен к выбросам, которые присутствуют в fare_amount и trip_distance. 

Пропуски в passenger_count заполнены единицей, так как поездка не может быть без пассажиров.

Надбавочные признаки (extra, mta_tax и др.) закодированы порядково - их значения можно естественно упорядочить по возрастанию.

Категориальные признаки с небольшим числом категорий (VendorID, RatecodeID, payment_type) преобразованы через one-hot кодирование.

Для времени и локаций разработаны кастомные трансформеры, позволившие извлечь максимум информации (день недели, час, частоты маршрутов) без создания чрезмерно большоко количества новых признаков.



### 7. ColumnTransformer

In [26]:
preprocessor = ColumnTransformer([
    ('datetime', Pipeline([
        ('extractor', DateTimeExtractor())
    ]), ['tpep_pickup_datetime']),
    
    ('locations', Pipeline([
        ('encoder', LocationFrequencyEncoder())
    ]), ['PULocationID', 'DOLocationID']),
    
    ('continuous', continuous_pipeline, continuous_features),
    ('discrete', discrete_pipeline, discrete_features),
    ('surcharge', surcharge_pipeline, surcharge_features),
    ('categorical_small', categorical_small_pipeline, categorical_small),
], remainder='drop') 

ColumnTransformer объединяет все созданные пайплайны в единый процесс предобработки данных. Для каждой группы признаков применяется соответствующий трансформер, а результаты будут конкатенироваться в итоговую матрицу признаков.

### 8. Применение пайплайна

In [27]:
train_processed = preprocessor.fit_transform(train_features)

feature_names = []

# Для datetime признаков
datetime_transformer = preprocessor.named_transformers_['datetime'].named_steps['extractor']
datetime_features = ['pickup_dayofweek', 'pickup_hour', 'pickup_month', 'pickup_day', 
                     'time_sin', 'time_cos']
feature_names.extend(datetime_features)

# Для location признаков
location_transformer = preprocessor.named_transformers_['locations'].named_steps['encoder']
location_features = ['PU_freq', 'DO_freq', 'location_pair_freq']
feature_names.extend(location_features)

# Для continuous признаков
feature_names.extend(continuous_features)

# Для discrete признаков
feature_names.extend(discrete_features)

# Для surcharge признаков 
surcharge_features_encoded = surcharge_features  
feature_names.extend(surcharge_features_encoded)

# Для categorical_small признаков 
categorical_small_transformer = preprocessor.named_transformers_['categorical_small'].named_steps['encoder']
categorical_small_encoded = categorical_small_transformer.get_feature_names_out(categorical_small)
feature_names.extend(categorical_small_encoded)

# Создаем DataFrame
train_processed_df = pd.DataFrame(
    train_processed,
    columns=feature_names,
    index=train_features.index
)

print(train_processed_df.head())
print(f"Размер обработанного DataFrame: {train_processed_df.shape}")

   pickup_dayofweek  pickup_hour  pickup_month  pickup_day  time_sin  \
0               0.0          0.0           1.0         1.0  0.250028   
1               0.0          0.0           1.0         1.0  0.013090   
2               0.0          0.0           1.0         1.0  0.074544   
3               0.0          0.0           1.0         1.0  0.159163   
4               0.0          0.0           1.0         1.0  0.203001   

   time_cos   PU_freq   DO_freq  location_pair_freq  trip_distance  ...  \
0  0.968239  0.000915  0.028581            0.021765       0.014085  ...   
1  0.999914  0.001945  0.021057            0.045924       0.051643  ...   
2  0.997218  0.000151  0.045924            0.021765       1.413146  ...   
3  0.987252  0.000371  0.021765            0.007699      -0.136150  ...   
4  0.979179  0.000289  0.007699            0.009830      -0.417840  ...   

   RatecodeID_2.0  RatecodeID_3.0  RatecodeID_4.0  RatecodeID_5.0  \
0             0.0             0.0             0

После применения пайплайна к обучающим данным с помощью fit_transform() получается numpy-массив, который затем преобразуется обратно в DataFrame с восстановленными названиями признаков для удобства анализа и интерпретации. 

In [18]:
train_processed_df.columns

Index(['pickup_dayofweek', 'pickup_hour', 'pickup_month', 'pickup_day',
       'time_sin', 'time_cos', 'PU_freq', 'DO_freq', 'location_pair_freq',
       'trip_distance', 'fare_amount', 'tip_amount', 'tolls_amount',
       'total_amount', 'congestion_surcharge', 'passenger_count', 'extra',
       'mta_tax', 'improvement_surcharge', 'Airport_fee', 'store_and_fwd_flag',
       'VendorID_1.0', 'VendorID_2.0', 'RatecodeID_1.0', 'RatecodeID_2.0',
       'RatecodeID_3.0', 'RatecodeID_4.0', 'RatecodeID_5.0', 'RatecodeID_6.0',
       'RatecodeID_99.0', 'payment_type_1.0', 'payment_type_2.0',
       'payment_type_3.0', 'payment_type_4.0'],
      dtype='object')

In [19]:
train_processed_df.head(10)

,pickup_dayofweek,pickup_hour,pickup_month,pickup_day,time_sin,time_cos,PU_freq,DO_freq,location_pair_freq,trip_distance,...,RatecodeID_2.0,RatecodeID_3.0,RatecodeID_4.0,RatecodeID_5.0,RatecodeID_6.0,RatecodeID_99.0,payment_type_1.0,payment_type_2.0,payment_type_3.0,payment_type_4.0
0,0.0,0.0,1.0,1.0,0.250028,0.968239,0.000915,0.028581,0.021765,0.014085,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,1.0,0.013090,0.999914,0.001945,0.021057,0.045924,0.051643,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.0,0.0,1.0,1.0,0.074544,0.997218,0.000151,0.045924,0.021765,1.413146,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0.0,0.0,1.0,1.0,0.159163,0.987252,0.000371,0.021765,0.007699,-0.136150,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.0,0.0,1.0,1.0,0.203001,0.979179,0.000289,0.007699,0.009830,-0.417840,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5,0.0,0.0,1.0,1.0,0.234011,0.972234,0.000146,0.009830,0.026165,1.413146,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
6,0.0,0.0,1.0,1.0,0.215303,0.976547,0.000285,0.021549,0.001437,4.286385,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
8,0.0,0.0,1.0,1.0,0.113275,0.993564,0.000178,0.043652,0.004523,1.760563,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
9,0.0,0.0,1.0,1.0,0.122447,0.992475,0.000397,0.013918,0.013918,-0.774648,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
10,0.0,0.0,1.0,1.0,0.153704,0.988117,0.000861,0.020323,0.011928,-0.441315,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


### 9. Обучение регрессионной модели

Для обучения модели, предсказывающей время в пути, некоторые признаки были признаны неинформативными и избыточными. Оставшиеся колонки: time_sin, time_cos, fare_amount, trip_distance, pickup_dayofweek.

In [20]:
X_features  =train_processed_df.drop([ 'pickup_hour', 'pickup_month', 'pickup_day', 'PU_freq',
       'DO_freq', 'location_pair_freq', 
       'tip_amount', 'tolls_amount', 'total_amount', 'congestion_surcharge',
       'passenger_count', 'extra', 'mta_tax', 'improvement_surcharge',
       'Airport_fee', 'store_and_fwd_flag', 'VendorID_1.0', 'VendorID_2.0',
       'RatecodeID_1.0', 'RatecodeID_2.0', 'RatecodeID_3.0', 'RatecodeID_4.0',
       'RatecodeID_5.0', 'RatecodeID_6.0', 'RatecodeID_99.0',
       'payment_type_1.0', 'payment_type_2.0', 'payment_type_3.0',
       'payment_type_4.0'],axis= 1)

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_features, 
    train_target,
    test_size=0.2,        
    random_state=42,      
    shuffle=True         
)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер валидационной выборки: {X_val.shape}")

Размер обучающей выборки: (6854536, 5)
Размер валидационной выборки: (1713635, 5)


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
rf_model = RandomForestRegressor(
    n_estimators=100,     
    max_depth=10,           
    random_state=42,
    n_jobs=-1,             
    verbose=1             
)

# Обучение
print("Обучение модели...")
rf_model.fit(X_train, y_train)

# Предсказание
y_pred = rf_model.predict(X_val)

# 5. Оценка
print("Результаты базовой модели:")
print(f"R² Score: {r2_score(y_val, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_val, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_pred)):.4f}")


Обучение модели...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:  7.3min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed: 16.0min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.2s


Результаты базовой модели:
R² Score: 0.8705
MAE: 101.5351
RMSE: 273.0023


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    5.0s finished


### 10. Применение к тестовому набору данных

In [28]:
test_processed = preprocessor.transform(val_features)
# Создаем DataFrame
test_processed_df = pd.DataFrame(
    test_processed,
    columns=feature_names,
    index=val_features.index
)

print(train_processed_df.head())
print(f"Размер обработанного DataFrame: {test_processed_df.shape}")

   pickup_dayofweek  pickup_hour  pickup_month  pickup_day  time_sin  \
0               0.0          0.0           1.0         1.0  0.250028   
1               0.0          0.0           1.0         1.0  0.013090   
2               0.0          0.0           1.0         1.0  0.074544   
3               0.0          0.0           1.0         1.0  0.159163   
4               0.0          0.0           1.0         1.0  0.203001   

   time_cos   PU_freq   DO_freq  location_pair_freq  trip_distance  ...  \
0  0.968239  0.000915  0.028581            0.021765       0.014085  ...   
1  0.999914  0.001945  0.021057            0.045924       0.051643  ...   
2  0.997218  0.000151  0.045924            0.021765       1.413146  ...   
3  0.987252  0.000371  0.021765            0.007699      -0.136150  ...   
4  0.979179  0.000289  0.007699            0.009830      -0.417840  ...   

   RatecodeID_2.0  RatecodeID_3.0  RatecodeID_4.0  RatecodeID_5.0  \
0             0.0             0.0             0

In [29]:
test_processed_df.head(20)

,pickup_dayofweek,pickup_hour,pickup_month,pickup_day,time_sin,time_cos,PU_freq,DO_freq,location_pair_freq,trip_distance,...,RatecodeID_2.0,RatecodeID_3.0,RatecodeID_4.0,RatecodeID_5.0,RatecodeID_6.0,RatecodeID_99.0,payment_type_1.0,payment_type_2.0,payment_type_3.0,payment_type_4.0
0,0.0,0.0,4.0,1.0,0.011635,0.999932,0.000080,0.043652,0.001449,1.647887,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,0.0,0.0,4.0,1.0,0.178802,0.983885,0.001813,0.004099,0.004099,1.835681,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.0,0.0,4.0,1.0,0.210898,0.977508,0.000636,0.028581,0.045924,0.873239,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0.0,0.0,4.0,1.0,0.242063,0.970261,0.000280,0.011928,0.021805,-0.295775,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,4.0,1.0,0.037225,0.999307,0.001630,0.045924,0.020581,-0.464789,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5,0.0,0.0,4.0,1.0,0.070409,0.997518,0.000837,0.031692,0.026160,-0.028169,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6,0.0,0.0,4.0,1.0,0.009381,0.999956,0.000092,0.021549,0.000588,1.413146,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
7,0.0,0.0,4.0,1.0,0.118693,0.992931,0.000290,0.028581,0.005537,0.727700,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
8,0.0,0.0,4.0,1.0,0.179804,0.983702,0.000289,0.028999,0.013368,9.300469,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
9,0.0,0.0,4.0,1.0,0.083098,0.996541,0.000506,0.020563,0.033199,0.328638,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


Модель random forest, обученная на отобранных признаках, показала приемлемые результаты: коэффициент детерминации R² составил 0.8705, средняя абсолютная ошибка - 101 секунды.

## Итоговый отчет

### (1) IEEE-CIS Fraud Detection — задача бинарной классификации мошеннических онлайн-транзакций:
1. Структура пайплайна: числа, содержащие мешающие выбросы -> номера и числа с важными аномалиями -> категории -> время -> разреженные числа -> разреженные категории.
2. Ключевые этапы предобработки:
- загрузить и вывести содержимое датасета;
- разбить признаки на подгруппы по типам данных и семантике;
- проанализировать подгруппы на предмет пропусков и аномалий;
- исходя из первичного анализа написать трансформеры для каждой подгруппы;
- собрать трансформеры в отдельные пайплайны для каждой подгруппы;
- объединить отдельные пайплайны для каждой подгруппы в один общий пайплайн предобработки данных;
- применить пайплайн к датасету.
3. Преобразование признаков:
- числовые признаки: вставка медианы в пропуски -> удаление выбросов методом межквартильного расстояния, если это необходимо -> стандартизация значений;
- категориальные признаки: вставка моды в пропуски -> преобразование строки в ее частоту среди всех значений в колонке -> вставка частоты 0 вместо невстреченных в train значений;
- временные признаки: вставка медианы в пропуски -> удаление выбросов методом межквартильного расстояния -> циклическое преобразование секунд в часы и дни недели;
- разреженные числовые признаки: вставка константы -999 в пропуски -> стандартизация значений;
- разреженные категориальные признаки: вставка константы "missing" в пропуски -> преобразование строки в ее частоту среди всех значений в колонке -> вставка частоты 0 вместо невстреченных в train значений.
4. Итоговая базовая модель и полученные результаты решения:
- Базовая модель - дерево решений средней глубины, чтобы оно не переобучилось, но в то же время успело изучить закономерности. Веса классов автоматически корректируются, чтобы учесть дисбаланс между классами.
- Полученные результаты - обученная модель отнесла небольшой процент транзакций к мошенническим, хотя таких в test data не было. Полученная точность равна примерно 0.83 и приемлема для такой базовой модели, как дерево решений.

### (2) NYC — задача регрессии длительности поездок
1. Структура пайплайна: преобразование и генерация новых временных признаков -> обработка ID локаций -> обработка непрерывных величин -> обработка дискретных величин -> обработка фиксированных надбавок -> работа с категориальными значениями 
2. Ключевые этапы предобработки:
- был загружен датасет;
- проанализированы признаки на предмет пропусков и выбросов;
- признаки были разбиты на подгруппы по типам данных и способу их последующей обработки;
- исходя из анализа были написаны или взяты из стандартной библиотеки sklearn трансформеры для каждой подгруппы;
- трансформеры объединены в отдельные пайплайны;
- создан один общий пайплайн предобработки данных;
- пайплайн был применен к датасету
3. Преобразование признаков:
- непрерывные признаки: на место пропусков осуществлена вставка медианы -> преобразование значений с помощью RobustScaler
- дискретные признаки: на место пропусков подставляется значение 1
- фиксированные надбавки: пропущенные значения заполняются наиболее часто встречаемым значением -> значения нумеруются в порядке возрастания
- категориальные признаки: пропуски заменяются наиболее часто встречаемым значением -> используется OneHotEncoder
- временные признаки: значение переводятся в тип datetime -> создаются более конкретные признаки (циклическое кодирование времени суток и определение дня недели)
- локации: был создан новый признак как комбинация места отбытия и места прибытия -> два изначальных и один новый признак проходят через частотное кодирование
4. Итоговая базовая модель и полученные результаты решения:
- Базовая модель - случайный лес с 100 деревьев и максимальной глубиной 10 для экономии памяти. 
- Обученная модель может объяснить 87% дисперсии длительности поездок. Средняя абсолютная ошибка составляет 101 секунду. Среднеквадратичная ошибка - 273 секунды . Результаты можно назвать приемлемыми 